# LOGO Metric Feasibility Warning

Leave-One-Group-Out (LOGO) is correct when the validation goal is generalization to unseen patients, sites, or subjects. The catch is metric feasibility: if a held-out group is small or contains only one outcome class, fold-wise ROC-AUC can be undefined and sensitivity/specificity can be unstable.

TrustCV keeps LOGO behavior unchanged. It warns and exposes diagnostics, then lets you compute pooled out-of-fold metrics explicitly.

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LogisticRegression

cwd = Path.cwd()
repo_root = cwd.parent if cwd.name == "notebooks" else cwd
if (repo_root / "trustcv").exists() and str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from trustcv import UniversalCVRunner, LeaveOneGroupOut
from trustcv.metrics import oob_clinical_metrics

warnings.simplefilter("always", UserWarning)

## Synthetic Repeated-Patient Dataset

There are six patients with two repeated records each. Each patient is single-class, so each LOGO validation fold contains only class 0 or only class 1.

In [ ]:
rng = np.random.default_rng(7)
patient_id = np.repeat(np.arange(6), 2)
patient_label = np.array([0, 0, 0, 1, 1, 1])
y = np.repeat(patient_label, 2)

X = np.column_stack([
    y + rng.normal(scale=0.05, size=len(y)),
    patient_id / patient_id.max(),
    rng.normal(size=len(y)),
])

data = pd.DataFrame(X, columns=["signal", "patient_scaled", "noise"])
data["patient_id"] = patient_id
data["outcome"] = y
display(data)

## Run LOGO With Fold-Wise AUC Requested

The warning is expected. LOGO still preserves group exclusivity, but each held-out patient has only one class, so fold-wise AUC is not feasible.

In [ ]:
runner = UniversalCVRunner(
    cv_splitter=LeaveOneGroupOut(),
    framework="sklearn",
    verbose=0,
)

results = runner.run(
    model=LogisticRegression(),
    data=(X, y),
    groups=patient_id,
    metrics=["roc_auc", "accuracy"],
)

print(results.summary())

## Diagnostic Table

The result object stores structured diagnostics. Each row describes one held-out fold.

In [ ]:
diagnostics = results.diagnostics["metric_feasibility"]
diagnostic_table = pd.DataFrame(diagnostics["folds"])
display(diagnostic_table[[
    "fold_display",
    "test_fold_size",
    "n_unique_classes",
    "class_counts",
    "roc_auc_feasible",
    "sensitivity_feasible",
    "specificity_feasible",
    "warning",
]])

print(diagnostics["recommendation"])

## Pooled Out-of-Fold AUC

Pooled OOF aggregation concatenates all held-out predictions first and then computes one global metric. This can be feasible even when every individual LOGO fold is single-class.

In [ ]:
pooled = oob_clinical_metrics(results, y)
pooled_summary = {
    "pooled_auc_roc": pooled.get("auc_roc"),
    "pooled_sensitivity": pooled.get("sensitivity"),
    "pooled_specificity": pooled.get("specificity"),
    "pooled_accuracy": pooled.get("accuracy"),
}
display(pd.DataFrame([pooled_summary]))